In [4]:
import tensorflow as tf
import datetime
import pickle
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential,load_model

In [5]:
## Load the trained model scaler pickle,onehot

model=load_model('model.h5')


##load the encoder and scaler 
with open('one_hot_encoder_geography.pkl','rb') as f:
    label_encoder_geo=pickle.load(f)

with open('label_encoder_gender.pkl','rb') as f:
    label_encoder_gender=pickle.load(f)

with open('scaler.pkl','rb') as f:
    scaler=pickle.load(f)

In [6]:
#Example input data for prediction
input_data={'CreditScore':600,
            'Geography':'France',
            'Gender':'Male',
            'Age':40,
            'Tenure':3,
            'Balance':60000,
            'NumOfProducts':2,
            'HasCrCard':1,
            'IsActiveMember':1,
            'EstimatedSalary':50000
            }

In [8]:
#One hot encode the categorical features
geo_encoded=label_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded,columns=label_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

f:\ANN CLASSIFICATION\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [9]:
input_data=pd.DataFrame([input_data])
input_data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [10]:
#encode the categorical features
input_data['Gender']=label_encoder_gender.transform(input_data[['Gender']])
input_data

f:\ANN CLASSIFICATION\venv\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [11]:
##concate with one hot encoded features
input_data=pd.concat([input_data.drop('Geography',axis=1),geo_encoded_df],axis=1)
input_data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [14]:
#scalling the input data 
scaled_input_data=scaler.transform(input_data)
scaled_input_data

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [15]:
#predict churn 
prediction=model.predict(scaled_input_data)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step


array([[0.01847213]], dtype=float32)

In [17]:
prediction_proba=prediction[0][0]
prediction_proba

np.float32(0.018472133)

In [18]:
if prediction_proba>0.5:
    print(f"The customer is likely to churn with a probability of {prediction_proba:.2f}") 
else:
    print(f"The customer is unlikely to churn with a probability of {prediction_proba:.2f}")

The customer is unlikely to churn with a probability of 0.02
